# 05H — Forward Temporal Generalization Audit

**Build ID:** `05H_V1_FORWARD_TEMPORAL_GENERALIZATION_AUDIT`

The frozen 05B/05C evaluation estimates generalization to unseen wildfire incidents, but its shuffled incident-grouped folds contain all years. This notebook adds a complementary expanding-window temporal audit:

- train on 2020, test on 2021;
- train on 2020–2021, test on 2022;
- train on 2020–2022, test on 2023;
- train on 2020–2023, test on 2024.

The primary model remains frozen. This notebook uses the exact ordered 803-feature set, ordinary row weighting, and exact fixed LightGBM parameter contract. It performs no tuning, calibration fitting, threshold selection, or feature selection.

Because 2021–2024 were already visible during prior model development, these results are a **post-hoc forward-temporal robustness analysis**, not pristine external validation. A future untouched year remains the strongest prospective validation design.

## 0. Configuration and scientific contract

In [ ]:
import os  # Configurable project root for the public release.
NOTEBOOK_BUILD_ID = "05H_V1_FORWARD_TEMPORAL_GENERALIZATION_AUDIT"

from pathlib import Path
import sys
import subprocess
import importlib.util

PROJECT_ROOT = Path(os.environ.get("FIRELINE_PROJECT_ROOT", "."))
MASTER_PATH = PROJECT_ROOT / "data" / "processed" / "modeling_table_assembly_v1" / "fireline_hold_modeling_master_v1.parquet"
UNIT_DESIGN_PATH = PROJECT_ROOT / "data" / "processed" / "modeling_design_v1" / "unit_outer_fold_and_training_weights_v1.parquet"
FEATURE_SETS_PATH = PROJECT_ROOT / "outputs" / "modeling_design_v1" / "FEATURE_SETS_V1.json"
PRIMARY_OOF_PATH = PROJECT_ROOT / "data" / "processed" / "baseline_modeling_v1" / "oof_predictions" / "E_FULL_EXTENDED__row.parquet"
PRIMARY_PARAMS_PATH = PROJECT_ROOT / "models" / "fireline_hold_primary_v1" / "fireline_hold_primary_v1_params.json"

OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "forward_temporal_generalization_v1"
PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed" / "forward_temporal_generalization_v1"
FIGURE_ROOT = OUTPUT_ROOT / "figures"
for p in [OUTPUT_ROOT, PROCESSED_ROOT, FIGURE_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

for p in [MASTER_PATH, UNIT_DESIGN_PATH, FEATURE_SETS_PATH, PRIMARY_OOF_PATH, PRIMARY_PARAMS_PATH]:
    assert p.exists(), p

EXPECTED_UNITS = 304_143
EXPECTED_INCIDENTS = 381
EXPECTED_YEAR_UNITS = {2020: 48_596, 2021: 101_517, 2022: 31_232, 2023: 10_467, 2024: 112_331}
EXPECTED_YEAR_INCIDENTS = {2020: 89, 2021: 87, 2022: 54, 2023: 47, 2024: 104}
TEST_YEARS = [2021, 2022, 2023, 2024]
MODEL_SEED = 20260825
BOOTSTRAP_REPS = 5000
BOOTSTRAP_SEED = 20260825
PRED_EPS = 1e-6
CALIBRATION_BINS = 10
MAX_NEWTON_ITER = 100
NEWTON_TOL = 1e-10
RIDGE = 1e-10

FIXED_PARAMS = {
    "objective": "binary",
    "n_estimators": 300,
    "learning_rate": 0.05,
    "num_leaves": 31,
    "max_depth": -1,
    "min_child_samples": 150,
    "subsample": 0.85,
    "colsample_bytree": 0.80,
    "reg_alpha": 0.0,
    "reg_lambda": 1.0,
    "max_bin": 127,
    "n_jobs": -1,
    "verbosity": -1,
}

required = {
    "pandas": "pandas",
    "numpy": "numpy",
    "pyarrow": "pyarrow",
    "sklearn": "scikit-learn",
    "scipy": "scipy",
    "lightgbm": "lightgbm",
    "matplotlib": "matplotlib",
}
for module, package in required.items():
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

from datetime import datetime, timezone
import gc
import json
import time

import numpy as np
import pandas as pd
from scipy.special import expit
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss, log_loss
from lightgbm import LGBMClassifier
import lightgbm as lgb
import sklearn
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 500)
pd.set_option("display.max_rows", 500)
pd.set_option("display.width", 420)

print("=" * 100)
print("RUNNING:", NOTEBOOK_BUILD_ID)
print("=" * 100)
print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__, "scikit-learn:", sklearn.__version__, "LightGBM:", lgb.__version__)

## 1. Load and verify the frozen model contract

The time split changes only which years are eligible for training. The feature order, target, model family, weighting, and fixed parameters remain frozen. `subsample_freq` must remain absent because it was absent from 05C.

In [ ]:
with FEATURE_SETS_PATH.open("r", encoding="utf-8") as f:
    feature_sets = json.load(f)
with PRIMARY_PARAMS_PATH.open("r", encoding="utf-8") as f:
    saved_primary_params = json.load(f)

features = feature_sets["E_FULL_EXTENDED"]
assert len(features) == 803
assert len(features) == len(set(features))
assert "subsample_freq" not in FIXED_PARAMS
assert saved_primary_params.get("random_state") == MODEL_SEED
assert {k: v for k, v in saved_primary_params.items() if k != "random_state"} == FIXED_PARAMS

design = pd.read_parquet(UNIT_DESIGN_PATH)
assert len(design) == EXPECTED_UNITS
assert design["unit_uid"].is_unique
assert design["_irwin_norm"].nunique() == EXPECTED_INCIDENTS
assert set(design["Year"].unique()) == set(EXPECTED_YEAR_UNITS)
assert (design.groupby("_irwin_norm")["Year"].nunique() == 1).all()

year_audit = (
    design.groupby("Year", as_index=False)
    .agg(units=("unit_uid", "size"), incidents=("_irwin_norm", "nunique"), held_rate=("hold_label", "mean"))
)
for row in year_audit.itertuples(index=False):
    assert row.units == EXPECTED_YEAR_UNITS[int(row.Year)]
    assert row.incidents == EXPECTED_YEAR_INCIDENTS[int(row.Year)]

split_rows = []
for test_year in TEST_YEARS:
    train = design[design["Year"] < test_year]
    test = design[design["Year"] == test_year]
    split_rows.append({
        "test_year": test_year,
        "train_start_year": int(train["Year"].min()),
        "train_end_year": int(train["Year"].max()),
        "train_units": len(train),
        "train_incidents": train["_irwin_norm"].nunique(),
        "train_held_rate": float(train["hold_label"].mean()),
        "test_units": len(test),
        "test_incidents": test["_irwin_norm"].nunique(),
        "test_held_rate": float(test["hold_label"].mean()),
    })

split_design = pd.DataFrame(split_rows)
assert (split_design["train_end_year"] < split_design["test_year"]).all()
display(year_audit)
display(split_design)
split_design.to_csv(OUTPUT_ROOT / "FORWARD_TEMPORAL_SPLIT_DESIGN_V1.csv", index=False)

model_contract = {
    "build_id": NOTEBOOK_BUILD_ID,
    "feature_set": "E_FULL_EXTENDED",
    "feature_count": len(features),
    "training_weighting": "ordinary row weighting",
    "parameters": {**FIXED_PARAMS, "random_state": MODEL_SEED},
    "test_years": TEST_YEARS,
    "temporal_rule": "For test year Y, training uses only Year < Y.",
    "classification_threshold": None,
    "posthoc_calibration": None,
    "interpretation": "Post-hoc temporal robustness analysis, not untouched external validation.",
}
(OUTPUT_ROOT / "FORWARD_TEMPORAL_MODEL_CONTRACT_V1.json").write_text(
    json.dumps(model_contract, indent=2), encoding="utf-8"
)
print("FROZEN CONTRACT: PASS")

## 2. Metric and calibration helpers

All test metrics are unweighted. Incident-macro probability losses include single-class incidents; incident-macro AUC metrics include mixed-class incidents only. Calibration intercept and slope use the stable Newton/IRLS implementation established in 05C1.

In [ ]:
def clip_prob(p):
    return np.clip(np.asarray(p, dtype=float), PRED_EPS, 1.0 - PRED_EPS)

def logit_prob(p):
    p = clip_prob(p)
    return np.log(p / (1.0 - p))

def binary_log_loss_vector(y, p):
    y = np.asarray(y, dtype=float)
    p = clip_prob(p)
    return -(y * np.log(p) + (1.0 - y) * np.log(1.0 - p))

def calibration_in_the_large(y, p):
    y = np.asarray(y, dtype=float)
    z = logit_prob(p)
    a = 0.0
    converged = False
    for iteration in range(MAX_NEWTON_ITER):
        mu = expit(a + z)
        step = np.sum(y - mu) / (np.sum(mu * (1.0 - mu)) + RIDGE)
        a += step
        if abs(step) < NEWTON_TOL:
            converged = True
            break
    return {"calibration_in_the_large": float(a), "citl_converged": bool(converged)}

def calibration_joint_intercept_slope(y, p):
    y = np.asarray(y, dtype=float)
    z = logit_prob(p)
    X = np.column_stack([np.ones_like(z), z])
    beta = np.array([0.0, 1.0], dtype=float)
    converged = False
    for iteration in range(MAX_NEWTON_ITER):
        mu = expit(X @ beta)
        w = mu * (1.0 - mu)
        information = X.T @ (X * w[:, None]) + RIDGE * np.eye(2)
        step = np.linalg.solve(information, X.T @ (y - mu))
        beta += step
        if np.max(np.abs(step)) < NEWTON_TOL:
            converged = True
            break
    return {
        "calibration_joint_intercept": float(beta[0]),
        "calibration_slope": float(beta[1]),
        "joint_converged": bool(converged),
    }

def per_incident_metrics(frame):
    rows = []
    for incident, g in frame.groupby("_irwin_norm", sort=False):
        y = g["hold_label"].to_numpy(int)
        p = g["pred_prob"].to_numpy(float)
        losses = binary_log_loss_vector(y, p)
        mixed = np.unique(y).size == 2
        rows.append({
            "_irwin_norm": incident,
            "units": len(g),
            "brier_sum": float(np.sum((p - y) ** 2)),
            "logloss_sum": float(np.sum(losses)),
            "brier": float(np.mean((p - y) ** 2)),
            "log_loss": float(np.mean(losses)),
            "mixed_class": mixed,
            "pr_auc": float(average_precision_score(y, p)) if mixed else np.nan,
            "roc_auc": float(roc_auc_score(y, p)) if mixed else np.nan,
        })
    return pd.DataFrame(rows)

def reliability_table(frame, bins=CALIBRATION_BINS):
    x = frame[["hold_label", "pred_prob"]].copy()
    x["bin"] = pd.cut(x["pred_prob"], bins=np.linspace(0.0, 1.0, bins + 1), include_lowest=True)
    rel = (
        x.groupby("bin", observed=True, as_index=False)
        .agg(units=("hold_label", "size"), mean_pred=("pred_prob", "mean"), observed_rate=("hold_label", "mean"))
    )
    rel["abs_gap"] = (rel["mean_pred"] - rel["observed_rate"]).abs()
    rel["weight"] = rel["units"] / rel["units"].sum()
    return rel, float((rel["weight"] * rel["abs_gap"]).sum())

def summarize_predictions(frame):
    y = frame["hold_label"].to_numpy(int)
    p = frame["pred_prob"].to_numpy(float)
    assert np.unique(y).size == 2
    inc = per_incident_metrics(frame)
    mixed = inc["mixed_class"]
    rel, ece = reliability_table(frame)
    metrics = {
        "units": len(frame),
        "incidents": frame["_irwin_norm"].nunique(),
        "held_rate": float(np.mean(y)),
        "mean_pred": float(np.mean(p)),
        "prediction_bias": float(np.mean(p) - np.mean(y)),
        "pooled_pr_auc": float(average_precision_score(y, p)),
        "pooled_roc_auc": float(roc_auc_score(y, p)),
        "pooled_brier": float(brier_score_loss(y, p)),
        "pooled_log_loss": float(log_loss(y, clip_prob(p), labels=[0, 1])),
        "incident_macro_brier": float(inc["brier"].mean()),
        "incident_macro_log_loss": float(inc["log_loss"].mean()),
        "mixed_incidents": int(mixed.sum()),
        "mixed_incident_macro_pr_auc": float(inc.loc[mixed, "pr_auc"].mean()),
        "mixed_incident_macro_roc_auc": float(inc.loc[mixed, "roc_auc"].mean()),
        "ece_fixed_width": ece,
        **calibration_in_the_large(y, p),
        **calibration_joint_intercept_slope(y, p),
    }
    return metrics, inc, rel

print("METRIC HELPERS: READY")

## 3. Fit the four frozen expanding-window models

Each test year is predicted only by a model trained on earlier years. The historical-prevalence prior uses the same earlier-year training rows. The saved mixed-year 05C OOF probability is retained only as a descriptive reference and is explicitly not a forward-time prediction.

In [ ]:
feature_table = pd.read_parquet(MASTER_PATH, columns=["unit_uid", *features])
assert len(feature_table) == EXPECTED_UNITS
assert feature_table["unit_uid"].is_unique

base_columns = ["unit_uid", "_irwin_norm", "IncidentName", "Year", "hold_label", "outer_fold"]
frame = design[base_columns].merge(feature_table, on="unit_uid", how="left", validate="one_to_one")
del feature_table
gc.collect()

X = frame[features].to_numpy(dtype=np.float32, na_value=np.nan)
y = frame["hold_label"].to_numpy(np.int8)
years = frame["Year"].to_numpy(int)
forward_pred = np.full(len(frame), np.nan, dtype=np.float64)
historical_prior = np.full(len(frame), np.nan, dtype=np.float64)
runtime_rows = []

for test_year in TEST_YEARS:
    train_idx = np.flatnonzero(years < test_year)
    test_idx = np.flatnonzero(years == test_year)
    assert years[train_idx].max() < test_year
    assert set(frame.iloc[train_idx]["_irwin_norm"]).isdisjoint(set(frame.iloc[test_idx]["_irwin_norm"]))

    params = dict(FIXED_PARAMS)
    params["random_state"] = MODEL_SEED
    model = LGBMClassifier(**params)

    started = time.time()
    model.fit(X[train_idx], y[train_idx])
    fit_seconds = time.time() - started

    forward_pred[test_idx] = clip_prob(model.predict_proba(X[test_idx])[:, 1])
    historical_prior[test_idx] = float(np.mean(y[train_idx]))

    runtime_rows.append({
        "test_year": test_year,
        "train_start_year": int(years[train_idx].min()),
        "train_end_year": int(years[train_idx].max()),
        "train_units": len(train_idx),
        "train_incidents": frame.iloc[train_idx]["_irwin_norm"].nunique(),
        "test_units": len(test_idx),
        "test_incidents": frame.iloc[test_idx]["_irwin_norm"].nunique(),
        "features": len(features),
        "fit_seconds": fit_seconds,
    })
    print(f"Test {test_year}: train through {test_year - 1} | fit={fit_seconds/60:.3f} min")
    del model
    gc.collect()

runtime = pd.DataFrame(runtime_rows)
runtime.to_csv(OUTPUT_ROOT / "FORWARD_TEMPORAL_RUNTIME_V1.csv", index=False)

primary_saved = pd.read_parquet(PRIMARY_OOF_PATH, columns=["unit_uid", "pred_prob"])
assert len(primary_saved) == EXPECTED_UNITS
assert primary_saved["unit_uid"].is_unique
primary_saved = primary_saved.rename(columns={"pred_prob": "pred_mixed_year_oof_reference"})

prediction_table = frame.loc[years >= min(TEST_YEARS), base_columns].copy()
prediction_table["pred_forward_fixed_05c"] = forward_pred[years >= min(TEST_YEARS)]
prediction_table["pred_historical_prevalence_prior"] = historical_prior[years >= min(TEST_YEARS)]
prediction_table = prediction_table.merge(primary_saved, on="unit_uid", how="left", validate="one_to_one")

prediction_columns = [
    "pred_forward_fixed_05c",
    "pred_historical_prevalence_prior",
    "pred_mixed_year_oof_reference",
]
assert np.isfinite(prediction_table[prediction_columns].to_numpy(float)).all()
assert ((prediction_table[prediction_columns] > 0) & (prediction_table[prediction_columns] < 1)).all().all()

PREDICTION_PATH = PROCESSED_ROOT / "FORWARD_TEMPORAL_PREDICTIONS_V1.parquet"
prediction_table.to_parquet(PREDICTION_PATH, index=False)
display(runtime)
print("Saved:", PREDICTION_PATH)

del X
gc.collect()

## 4. Year-specific temporal metrics

Three probability sets are shown for each test year: the forward frozen model, a historical-training prevalence prior, and the already-saved mixed-year incident-held-out 05C OOF reference. Only the first two obey forward chronology.

In [ ]:
MODEL_COLUMNS = {
    "FORWARD_FIXED_05C": "pred_forward_fixed_05c",
    "HISTORICAL_PREVALENCE_PRIOR": "pred_historical_prevalence_prior",
    "MIXED_YEAR_PRIMARY_OOF_REFERENCE": "pred_mixed_year_oof_reference",
}

metric_rows = []
incident_tables = {}
reliability_frames = []

for test_year in TEST_YEARS:
    year_frame = prediction_table[prediction_table["Year"] == test_year]
    for model_id, pred_col in MODEL_COLUMNS.items():
        evaluated = year_frame[base_columns].copy()
        evaluated["pred_prob"] = year_frame[pred_col].to_numpy(float)
        summary, inc, rel = summarize_predictions(evaluated)
        metric_rows.append({"test_year": test_year, "model_id": model_id, **summary})
        incident_tables[(test_year, model_id)] = inc
        rel["test_year"] = test_year
        rel["model_id"] = model_id
        rel["bin"] = rel["bin"].astype(str)
        reliability_frames.append(rel)

metrics = pd.DataFrame(metric_rows)
reliability = pd.concat(reliability_frames, ignore_index=True)
metrics.to_csv(OUTPUT_ROOT / "FORWARD_TEMPORAL_METRICS_V1.csv", index=False)
reliability.to_csv(OUTPUT_ROOT / "FORWARD_TEMPORAL_RELIABILITY_V1.csv", index=False)

display(metrics[[
    "test_year", "model_id", "units", "incidents", "held_rate", "mean_pred",
    "pooled_pr_auc", "pooled_roc_auc", "pooled_brier", "pooled_log_loss",
    "incident_macro_brier", "incident_macro_log_loss",
    "mixed_incident_macro_pr_auc", "mixed_incident_macro_roc_auc",
    "calibration_in_the_large", "calibration_slope", "ece_fixed_width",
]])

## 5. Paired wildfire-cluster bootstrap

For each test year, wildfire incidents are resampled with replacement 5,000 times. The same sampled incident multiplicities are applied to both models in a comparison. Deltas are `FORWARD_FIXED_05C − comparator`; negative is better for probability-loss metrics and positive is better for AUC metrics.

In [ ]:
BOOTSTRAP_METRICS = [
    "pooled_brier",
    "pooled_log_loss",
    "incident_macro_brier",
    "incident_macro_log_loss",
    "mixed_incident_macro_pr_auc",
    "mixed_incident_macro_roc_auc",
]
LOSS_METRICS = {"pooled_brier", "pooled_log_loss", "incident_macro_brier", "incident_macro_log_loss"}
COMPARATORS = ["HISTORICAL_PREVALENCE_PRIOR", "MIXED_YEAR_PRIMARY_OOF_REFERENCE"]

def bootstrap_metrics_from_incident_table(x, multiplicity):
    units = x["units"].to_numpy(float)
    total_units = np.sum(multiplicity * units)
    mixed = x["mixed_class"].to_numpy(bool)
    mixed_mult = multiplicity[mixed]
    return {
        "pooled_brier": float(np.sum(multiplicity * x["brier_sum"].to_numpy(float)) / total_units),
        "pooled_log_loss": float(np.sum(multiplicity * x["logloss_sum"].to_numpy(float)) / total_units),
        "incident_macro_brier": float(np.sum(multiplicity * x["brier"].to_numpy(float)) / np.sum(multiplicity)),
        "incident_macro_log_loss": float(np.sum(multiplicity * x["log_loss"].to_numpy(float)) / np.sum(multiplicity)),
        "mixed_incident_macro_pr_auc": float(np.sum(mixed_mult * x.loc[mixed, "pr_auc"].to_numpy(float)) / np.sum(mixed_mult)),
        "mixed_incident_macro_roc_auc": float(np.sum(mixed_mult * x.loc[mixed, "roc_auc"].to_numpy(float)) / np.sum(mixed_mult)),
    }

bootstrap_rows = []
for test_year in TEST_YEARS:
    incident_ids = np.array(sorted(incident_tables[(test_year, "FORWARD_FIXED_05C")]["_irwin_norm"].unique()))
    aligned = {}
    for model_id in MODEL_COLUMNS:
        x = incident_tables[(test_year, model_id)].set_index("_irwin_norm").loc[incident_ids]
        assert list(x.index) == list(incident_ids)
        aligned[model_id] = x

    rng = np.random.default_rng(BOOTSTRAP_SEED + test_year)
    n_incidents = len(incident_ids)
    for rep in range(BOOTSTRAP_REPS):
        draw = rng.integers(0, n_incidents, size=n_incidents)
        multiplicity = np.bincount(draw, minlength=n_incidents).astype(float)
        forward_rep = bootstrap_metrics_from_incident_table(aligned["FORWARD_FIXED_05C"], multiplicity)
        for comparator in COMPARATORS:
            comparator_rep = bootstrap_metrics_from_incident_table(aligned[comparator], multiplicity)
            for metric in BOOTSTRAP_METRICS:
                bootstrap_rows.append({
                    "test_year": test_year,
                    "comparison_id": f"FORWARD_FIXED_05C_MINUS_{comparator}",
                    "rep": rep,
                    "metric": metric,
                    "delta_forward_minus_comparator": forward_rep[metric] - comparator_rep[metric],
                })

bootstrap = pd.DataFrame(bootstrap_rows)
bootstrap_summary_rows = []
for (test_year, comparison_id, metric), g in bootstrap.groupby(["test_year", "comparison_id", "metric"], sort=False):
    d = g["delta_forward_minus_comparator"].to_numpy(float)
    better = d < 0 if metric in LOSS_METRICS else d > 0
    bootstrap_summary_rows.append({
        "test_year": test_year,
        "comparison_id": comparison_id,
        "metric": metric,
        "mean_delta_forward_minus_comparator": float(np.mean(d)),
        "ci_low": float(np.quantile(d, 0.025)),
        "ci_high": float(np.quantile(d, 0.975)),
        "probability_forward_better": float(np.mean(better)),
    })

bootstrap_summary = pd.DataFrame(bootstrap_summary_rows)
bootstrap_summary.to_csv(OUTPUT_ROOT / "FORWARD_TEMPORAL_PAIRED_BOOTSTRAP_V1.csv", index=False)
display(bootstrap_summary)

## 6. Figures, integrity checks, and report

The figures are audit outputs suitable for later refinement in the publication-figure notebook. They should not be interpreted as evidence from an untouched future cohort.

In [ ]:
MODEL_LABELS = {
    "FORWARD_FIXED_05C": "Forward fixed 05C",
    "HISTORICAL_PREVALENCE_PRIOR": "Historical prevalence prior",
    "MIXED_YEAR_PRIMARY_OOF_REFERENCE": "Mixed-year OOF reference",
}
COLORS = {
    "FORWARD_FIXED_05C": "#c44e52",
    "HISTORICAL_PREVALENCE_PRIOR": "#8c8c8c",
    "MIXED_YEAR_PRIMARY_OOF_REFERENCE": "#4c72b0",
}

plot_specs = [
    ("pooled_roc_auc", "Pooled ROC-AUC"),
    ("pooled_brier", "Pooled Brier (lower is better)"),
    ("mixed_incident_macro_roc_auc", "Mixed-incident macro ROC-AUC"),
    ("incident_macro_brier", "Incident-macro Brier (lower is better)"),
]
fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
for ax, (metric, title) in zip(axes.ravel(), plot_specs):
    for model_id in MODEL_COLUMNS:
        g = metrics[metrics["model_id"] == model_id].sort_values("test_year")
        ax.plot(g["test_year"], g[metric], marker="o", linewidth=2, label=MODEL_LABELS[model_id], color=COLORS[model_id])
    ax.set_title(title)
    ax.set_xlabel("Forward test year")
    ax.set_xticks(TEST_YEARS)
    ax.grid(alpha=0.25)
axes[0, 0].legend(frameon=False, fontsize=9)
fig.suptitle("Forward temporal generalization audit", fontsize=15)
metrics_png = FIGURE_ROOT / "FORWARD_TEMPORAL_METRICS_V1.png"
metrics_pdf = FIGURE_ROOT / "FORWARD_TEMPORAL_METRICS_V1.pdf"
fig.savefig(metrics_png, dpi=220, bbox_inches="tight")
fig.savefig(metrics_pdf, bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(10, 9), constrained_layout=True)
for ax, test_year in zip(axes.ravel(), TEST_YEARS):
    g = reliability[(reliability["test_year"] == test_year) & (reliability["model_id"] == "FORWARD_FIXED_05C")]
    ax.plot([0, 1], [0, 1], linestyle="--", color="black", linewidth=1)
    ax.plot(g["mean_pred"], g["observed_rate"], marker="o", color=COLORS["FORWARD_FIXED_05C"], linewidth=2)
    ax.set_title(f"Test year {test_year}")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_xlabel("Mean predicted probability")
    ax.set_ylabel("Observed Held rate")
    ax.grid(alpha=0.25)
fig.suptitle("Forward-model reliability by test year", fontsize=15)
reliability_png = FIGURE_ROOT / "FORWARD_TEMPORAL_RELIABILITY_V1.png"
reliability_pdf = FIGURE_ROOT / "FORWARD_TEMPORAL_RELIABILITY_V1.pdf"
fig.savefig(reliability_png, dpi=220, bbox_inches="tight")
fig.savefig(reliability_pdf, bbox_inches="tight")
plt.show()

expected_prediction_units = sum(EXPECTED_YEAR_UNITS[y] for y in TEST_YEARS)
bootstrap_group_sizes = bootstrap.groupby(["test_year", "comparison_id", "metric"]).size()
checks = pd.DataFrame([
    {"check": "exact_803_frozen_features", "pass": len(features) == 803},
    {"check": "exact_saved_primary_parameter_contract", "pass": {k: v for k, v in saved_primary_params.items() if k != "random_state"} == FIXED_PARAMS},
    {"check": "subsample_freq_remains_absent", "pass": "subsample_freq" not in FIXED_PARAMS},
    {"check": "exact_four_expanding_window_fits", "pass": len(runtime) == 4},
    {"check": "all_training_years_strictly_precede_test_year", "pass": bool((runtime["train_end_year"] < runtime["test_year"]).all())},
    {"check": "all_test_year_unit_counts_exact", "pass": all(len(prediction_table[prediction_table["Year"] == y]) == EXPECTED_YEAR_UNITS[y] for y in TEST_YEARS)},
    {"check": "all_test_year_incident_counts_exact", "pass": all(prediction_table.loc[prediction_table["Year"] == y, "_irwin_norm"].nunique() == EXPECTED_YEAR_INCIDENTS[y] for y in TEST_YEARS)},
    {"check": "prediction_table_exact_expected_units", "pass": len(prediction_table) == expected_prediction_units},
    {"check": "prediction_table_unique_units", "pass": prediction_table["unit_uid"].is_unique},
    {"check": "all_probability_sets_finite", "pass": bool(np.isfinite(prediction_table[prediction_columns].to_numpy(float)).all())},
    {"check": "all_probability_sets_in_open_unit_interval", "pass": bool(((prediction_table[prediction_columns] > 0) & (prediction_table[prediction_columns] < 1)).all().all())},
    {"check": "paired_incident_bootstrap_5000_each", "pass": bool((bootstrap_group_sizes == BOOTSTRAP_REPS).all())},
    {"check": "ordinary_row_weighting_only", "pass": True},
    {"check": "no_hyperparameter_tuning", "pass": True},
    {"check": "no_feature_selection", "pass": True},
    {"check": "no_posthoc_calibration_fitting", "pass": True},
    {"check": "no_threshold_selection", "pass": True},
    {"check": "posthoc_not_external_validation_label_frozen", "pass": True},
])
display(checks)
PASS = bool(checks["pass"].all())
checks.to_csv(OUTPUT_ROOT / "FINAL_FORWARD_TEMPORAL_CHECKS_V1.csv", index=False)

def md_table(df):
    try:
        return df.to_markdown(index=False)
    except Exception:
        return df.to_string(index=False)

report_lines = [
    "# 05H Forward Temporal Generalization Audit",
    "",
    f"Build: `{NOTEBOOK_BUILD_ID}`",
    "",
    f"Generated UTC: {datetime.now(timezone.utc).isoformat()}",
    "",
    "## Scientific status",
    "",
    "Post-hoc forward-temporal robustness analysis. This is not untouched external validation because the test years were visible during earlier model development.",
    "",
    "The frozen 05C primary model remains unchanged; 05H performs no model selection.",
    "",
    "## Expanding-window design",
    "",
    md_table(split_design),
    "",
    "## Point metrics",
    "",
    md_table(metrics),
    "",
    "## Paired wildfire-cluster bootstrap",
    "",
    md_table(bootstrap_summary),
    "",
    "## Runtime",
    "",
    md_table(runtime),
    "",
    "## Final checks",
    "",
    md_table(checks),
    "",
    "## Decision",
    "",
    "`PASS / FORWARD TEMPORAL AUDIT COMPLETE; READY FOR FROZEN-MODEL INTERPRETATION`" if PASS else "`REVIEW REQUIRED`",
]
REPORT_PATH = OUTPUT_ROOT / "FORWARD_TEMPORAL_GENERALIZATION_REPORT_V1.md"
REPORT_PATH.write_text("\n".join(report_lines), encoding="utf-8")

print()
print("05H FORWARD TEMPORAL GENERALIZATION:", "PASS" if PASS else "REVIEW REQUIRED")
print("Saved:", REPORT_PATH)
print("Saved figures:", metrics_png, reliability_png)
print()
print("DONE:", NOTEBOOK_BUILD_ID)

## 7. Output inventory

In [ ]:
print("=" * 100)
print("05H OUTPUTS")
print("=" * 100)
for root in [OUTPUT_ROOT, PROCESSED_ROOT]:
    print()
    print(root)
    for p in sorted(root.rglob("*")):
        if p.is_file():
            print(f"  {p.relative_to(root)!s:<80} {p.stat().st_size/(1024**2):>10.3f} MB")